# MMAction2 — fine-tune nhánh cảnh báo, giữ nguyên nhãn AVA
Backbone và đầu AVA gốc được khóa hoàn toàn. Chỉ huấn luyện nhánh tương tác MIL trên các khoảng thời gian public AIRTLab/RLVS đã duyệt bằng AI, không gán nhãn đánh nhau cho mọi người trong khung. Chưa có AVA đầy đủ trên Drive: đây không phải fine-tune toàn bộ model hoặc nghiệm thu camera. Không dùng/tải video iPhone riêng. Train/development/calibration/test giữ theo nhóm nguồn gốc. Nhãn gốc giữ nguyên; nhánh mới chỉ dự đoán cảnh báo đánh nhau, không tự tạo nhãn đá theo người.

Tối đa 45 phút GPU cả cài đặt và đánh giá; lưu cache/checkpoint/báo cáo trên Drive rồi tự ngắt. Không tự thay checkpoint mặc định của web.

In [ ]:
import time, threading
from pathlib import Path
from google.colab import drive, runtime
import torch
assert torch.cuda.is_available(), 'Hãy chọn GPU trong Runtime > Change runtime type.'
if 'MMACTION_DEADLINE' not in globals():
    MMACTION_DEADLINE = time.monotonic()+45*60
    watchdog = threading.Timer(45*60, runtime.unassign)
    watchdog.daemon = True
    watchdog.start()
drive.mount('/content/drive', timeout_ms=180000)
ROOT = Path('/content/drive/MyDrive/SENTINEL')
OUTPUT = ROOT/'artifacts/mmaction_interaction_v1'
OUTPUT.mkdir(parents=True,exist_ok=True)
def remaining(reserve=0):
    seconds=MMACTION_DEADLINE-time.monotonic()-reserve
    if seconds<=0: raise TimeoutError('Hết ngân sách GPU; file trên Drive giữ nguyên.')
    return seconds
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
import json, os, sys, subprocess, urllib.request, zipfile, shutil
try:
    with urllib.request.urlopen('https://api.github.com/repos/TranDuon/anomaly-behavior-detector-system/commits/main',timeout=60) as response:
        REVISION=json.load(response)['sha']
    assert len(REVISION)==40 and all(c in '0123456789abcdef' for c in REVISION)
    archive=Path('/content/fightwatch-mmaction-source.zip')
    with urllib.request.urlopen('https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/'+REVISION,timeout=90) as source, archive.open('wb') as target:
        shutil.copyfileobj(source,target)
    workspace=Path('/content/fightwatch-mmaction-source')
    workspace.mkdir(exist_ok=True)
    def extract_safe(archive_path,destination):
        with zipfile.ZipFile(archive_path) as package:
            assert all((destination/item.filename).resolve().is_relative_to(destination.resolve()) for item in package.infolist())
            package.extractall(destination)
    extract_safe(archive,workspace)
    CODE=workspace/('anomaly-behavior-detector-system-'+REVISION)
    os.chdir(CODE)
    sys.path.insert(0,str(CODE))
    from scripts.colab_process import run_logged
    def command(args,log,cap=600):
        return run_logged(args,OUTPUT/log,min(cap,remaining(180)),revision=REVISION)
    command([sys.executable,'-m','pip','install','-q','uv==0.8.22'],'uv_install.log',180)
    command(['uv','python','install','3.10'],'python_install.log',300)
    command(['uv','venv','--python','3.10','--seed','/content/mmaction-env'],'venv.log',180)
    PY='/content/mmaction-env/bin/python'
    command([PY,'-m','pip','install','-q','numpy==1.26.4','setuptools==69.5.1','wheel'],'dependencies.log')
    command([PY,'-m','pip','install','-q','torch==2.0.1+cu118','torchvision==0.15.2+cu118','--index-url','https://download.pytorch.org/whl/cu118'],'torch_install.log',600)
    command([PY,'-m','pip','install','-q','mmcv==2.1.0','--only-binary=mmcv','-f','https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html','mmengine==0.10.7','mmdet==3.2.0','decord==0.6.0','einops','importlib-metadata','scikit-learn==1.3.2','opencv-python==4.8.1.78'],'dependencies.log',600)
    DEMO=CODE/'mmaction2_demo'
    command(['git','clone','--depth','1','--branch','v1.2.0','https://github.com/open-mmlab/mmaction2.git',str(DEMO/'repo')],'upstream.log',180)
    commit=subprocess.check_output(['git','-C',str(DEMO/'repo'),'rev-parse','HEAD'],text=True).strip()
    assert commit=='4d6c93474730cad2f25e51109adcf96824efc7a3'
    command([PY,'-m','pip','install','--no-deps','--no-build-isolation','-e',str(DEMO/'repo')],'mmaction_install.log',180)
    print('Code:',REVISION)
except BaseException:
    runtime.unassign()
    raise


In [ ]:
import hashlib
CACHE=Path('/content/mmaction-feature-cache')
PACK=Path('/content/mmaction-public-review')
WEIGHTS=Path('/content/mmaction-public-weights')
try:
    for folder in (CACHE,PACK,WEIGHTS): folder.mkdir(exist_ok=True)
    package=ROOT/'artifacts/fight_temporal_review_v1/temporal_review_pack.zip'
    assert package.is_file(), 'Thiếu gói public temporal_review_pack.zip trên Drive'
    extract_safe(package,PACK)
    if (OUTPUT/'feature_cache.zip').exists():
        extract_safe(OUTPUT/'feature_cache.zip',CACHE)
    public_weights=OUTPUT/'public_weights'
    public_weights.mkdir(exist_ok=True)
    checkpoints={
      'action_slowonly_r101.pth':('https://download.openmmlab.com/mmaction/detection/ava/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb_20201217-16378594.pth','163785949c5178484012a6593d12d6fcf4d36be404fa42b5899b314e91f4703b'),
      'person_faster_rcnn_r50.pth':('https://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth','a5d8aa15c75b961b9c73f28a6e92af370835639a2939ff890852b052a5256a5b')}
    for name,(url,expected) in checkpoints.items():
        target=WEIGHTS/name
        if (public_weights/name).exists(): shutil.copyfile(public_weights/name,target)
        else:
            with urllib.request.urlopen(url,timeout=120) as source,target.open('wb') as destination:
                shutil.copyfileobj(source,destination)
        assert hashlib.sha256(target.read_bytes()).hexdigest()==expected, 'Checkpoint SHA mismatch'
        if not (public_weights/name).exists(): shutil.copyfile(target,public_weights/name)
    env=os.environ.copy()
    env['PYTHONPATH']=str(CODE)+os.pathsep+str(DEMO)+os.pathsep+str(DEMO/'repo')
    env['CUDA_MODULE_LOADING']='LAZY'
    env['OMP_NUM_THREADS']='4'
    os.environ.update(env)
    minutes=min(30,remaining(180)/60)
    assert minutes>=1
    run_logged([PY,str(DEMO/'train_interaction.py'),'--manifest',str(CODE/'benchmarks/mmaction_interaction_public_v1.json'),'--pack',str(PACK),'--cache',str(CACHE),'--weights',str(WEIGHTS),'--output',str(OUTPUT),'--minutes',str(minutes),'--epochs','120'],OUTPUT/'train.log',min(minutes*60+20,remaining(150)),revision=REVISION)
    print(json.dumps(json.loads((OUTPUT/'report.json').read_text()),ensure_ascii=False,indent=2))
    print('Gói nhánh mới:',OUTPUT/'mmaction_interaction_pilot.zip')
finally:
    try:
        partial=OUTPUT/'feature_cache.partial.zip'
        with zipfile.ZipFile(partial,'w',compression=zipfile.ZIP_STORED) as archive:
            for path in CACHE.glob('*.npz'):
                if not path.name.endswith('.partial.npz'): archive.write(path,path.name)
        partial.replace(OUTPUT/'feature_cache.zip')
    finally:
        print('Đã lưu cache/tiến độ. Ngắt GPU; không thay model mặc định trên web.',flush=True)
        runtime.unassign()
